In [39]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
print("TensorFlow version:", tf.__version__)
# Load the TensorBoard notebook extension
%load_ext tensorboard
import datetime
import shutil

TensorFlow version: 2.18.0
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [40]:
%reload_ext tensorboard
from tensorflow.keras import Model
from pathlib import Path
import pandas as pd

In [76]:
BATCH_SIZE = 128   
BUFFER_SIZE = 512   
LEARNING_RATE = 0.00001
EPOCHS = 4000 
# best results 

In [77]:
input_dir = Path('./prepared')
logs_path = Path('./logs')
if logs_path.exists():
  shutil.rmtree(logs_path) # удаляем, если существует /logs
logs_path.mkdir(parents=True)

X_train_name = input_dir / 'X_train.csv'
y_train_name = input_dir / 'y_train.csv'
X_test_name = input_dir / 'X_test.csv'
y_test_name = input_dir / 'y_test.csv'

X_train = pd.read_csv(X_train_name)
y_train = pd.read_csv(y_train_name)
X_test = pd.read_csv(X_test_name)
y_test = pd.read_csv(y_test_name)

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

In [78]:
@tf.keras.utils.register_keras_serializable() #  Декоратор позволяет сериализовать и десериализовать модель для сохранения и загрузки.
class SomeModel(Model):
    def __init__(self, neurons_cnt=64, **kwargs):
        super(SomeModel, self).__init__(**kwargs)
        self.neurons_cnt = neurons_cnt  # Сохраняем значение параметра для конфигурации
        self.d_in = Dense(30, activation='relu')
        self.d1 = Dense(neurons_cnt, activation='relu')
        self.d_out = Dense(1)

    def call(self, x):
        x = self.d_in(x)
        x = self.d1(x)
        return self.d_out(x)
         
    def build(self, input_shape): # надо явно определить для построения
        super(SomeModel, self).build(input_shape)
        
    def get_config(self): 
        # Возвращаем параметры модели, включая кастомные
        config = super(SomeModel, self).get_config()
        config.update({
            "neurons_cnt": self.neurons_cnt  # Добавляем кастомный параметр в конфигурацию
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Создаём экземпляр класса из конфигурации
        return cls(**config)

In [79]:
# Create an instance of the model
model = SomeModel(neurons_cnt=32)
model.build(input_shape=(None, 30))

In [80]:
loss_object = tf.keras.losses.MeanSquaredError() # что? 
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.MeanAbsoluteError(name='test_mae')

In [81]:
@tf.function
def train_step(input_vector, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(input_vector, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)

@tf.function
def test_step(input_vector, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(input_vector, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)

In [82]:
from tensorflow import keras
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = logs_path / 'gradient_tape' / current_time / 'train'
train_log_dir.mkdir(exist_ok=True, parents=True)
test_log_dir = logs_path / 'gradient_tape' / current_time / 'test'
test_log_dir.mkdir(exist_ok=True, parents=True)
train_summary_writer = tf.summary.create_file_writer(str(train_log_dir))
test_summary_writer = tf.summary.create_file_writer(str(test_log_dir))

logdir = logs_path / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir.mkdir(exist_ok=True, parents=True)
fit_summary_writer = tf.summary.create_file_writer(str(logdir))

tf.summary.trace_on(graph=True, profiler=True, profiler_outdir=str(logdir))

for epoch in range(EPOCHS):
  # Reset the metrics at the start of the next epoch
  for (x_train, y_train) in train_ds:

    with fit_summary_writer.as_default():
      train_step(x_train, y_train)


  with train_summary_writer.as_default():
    tf.summary.scalar('loss', train_loss.result(), step=epoch)
    tf.summary.scalar('accuracy', train_accuracy.result(), step=epoch)

  for (x_test, y_test) in test_ds:
    test_step(x_test, y_test)

  with test_summary_writer.as_default():
    tf.summary.scalar('loss', test_loss.result(), step=epoch)
    tf.summary.scalar('mae', test_accuracy.result(), step=epoch)
    for layer in model.layers:
        for weight in layer.weights:
            tf.summary.histogram(f"{layer.name}/{weight.name}", weight, step=epoch)
            
  template = 'Epoch {}, Loss: {}, Accuracy: {}, Test Loss: {}, Test MAE: {}'
  print (template.format(epoch+1,
                         train_loss.result(),
                         train_accuracy.result(),
                         test_loss.result(),
                         test_accuracy.result()))

  # Reset metrics every epoch
  train_loss.reset_state()
  test_loss.reset_state()
  train_accuracy.reset_state()
  test_accuracy.reset_state()

with fit_summary_writer.as_default():
  tf.summary.trace_export(
      name="my_func_trace",
      step=0,
      profiler_outdir=str(logdir)
  )

Epoch 1, Loss: 119.96746826171875, Accuracy: 10.893035888671875, Test Loss: 116.287841796875, Test MAE: 10.73458194732666
Epoch 2, Loss: 112.96160888671875, Accuracy: 10.558613777160645, Test Loss: 109.3155746459961, Test MAE: 10.40410041809082
Epoch 3, Loss: 105.89610290527344, Accuracy: 10.231470108032227, Test Loss: 102.68817138671875, Test MAE: 10.07975959777832
Epoch 4, Loss: 99.32025909423828, Accuracy: 9.909996032714844, Test Loss: 96.36974334716797, Test MAE: 9.760351181030273
Epoch 5, Loss: 93.03333282470703, Accuracy: 9.592233657836914, Test Loss: 90.31407928466797, Test MAE: 9.443937301635742
Epoch 6, Loss: 87.08012390136719, Accuracy: 9.2774019241333, Test Loss: 84.50542449951172, Test MAE: 9.13003158569336
Epoch 7, Loss: 81.55657196044922, Accuracy: 8.9650297164917, Test Loss: 78.93234252929688, Test MAE: 8.81823444366455
Epoch 8, Loss: 75.9271469116211, Accuracy: 8.654366493225098, Test Loss: 73.59352111816406, Test MAE: 8.50870132446289
Epoch 9, Loss: 70.56936645507812, 

In [83]:
%tensorboard --logdir ./logs/gradient_tape --port=8342

Reusing TensorBoard on port 8342 (pid 1828), started 0:14:38 ago. (Use '!kill 1828' to kill it.)

In [75]:
%load_ext tensorboard
%tensorboard --logdir=./logs/fit --port=8634

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 8634 (pid 26932), started 0:06:12 ago. (Use '!kill 26932' to kill it.)

In [61]:
tensorboard --logdir logs/weights/

Reusing TensorBoard on port 6006 (pid 16516), started 0:01:12 ago. (Use '!kill 16516' to kill it.)